# Entendimiento de datos: Base de videos de pádel

Este notebook explora `Data/Base_Videos_Final_202602.csv`: revisa sus variables, tipos, valores faltantes, distribuciones y algunos indicadores de calidad. El archivo es grande, así que el perfil completo se calcula por bloques para no cargarlo entero en memoria.

**Interpretación:** los nombres de las columnas permiten proponer una familia y una descripción inicial, pero no garantizan por sí solos el significado, la unidad ni la metodología de medición. Conviene contrastar estas hipótesis con quien generó los datos.

In [ ]:
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

RUTA_CSV = Path('Data') / 'Base_Videos_Final_202602.csv'
CHUNK_SIZE = 100_000

if not RUTA_CSV.is_file():
    raise FileNotFoundError(
        f'No se encontró el archivo: {RUTA_CSV.resolve()}. '
        'Ejecuta este notebook desde la raíz del repositorio.'
    )

print(f'Archivo: {RUTA_CSV.resolve()}')
print(f'Tamaño: {RUTA_CSV.stat().st_size / (1024 ** 2):,.1f} MB')
print(f'Tamaño de bloque para el perfil completo: {CHUNK_SIZE:,} filas')

## 1. Lectura inicial y muestra

La siguiente lectura carga sólo una muestra de 10.000 filas. Sirve para inspeccionar rápidamente el contenido y los tipos inferidos; las estadísticas de esta sección describen la muestra, no necesariamente todo el archivo.

In [ ]:
MUESTRA_FILAS = 10_000
muestra = pd.read_csv(
    RUTA_CSV,
    encoding='utf-8-sig',
    nrows=MUESTRA_FILAS,
    low_memory=False,
)

print(f'Filas en la muestra: {len(muestra):,}')
print(f'Variables: {muestra.shape[1]}')
display(muestra.head())
display(pd.DataFrame({
    'tipo inferido en muestra': muestra.dtypes.astype(str),
    'no nulos en muestra': muestra.notna().sum(),
    'faltantes en muestra': muestra.isna().sum(),
}).rename_axis('variable'))

## 2. Diccionario inicial de variables

Las familias y descripciones se infieren del nombre de cada columna. Los valores con coordenadas, longitudes o velocidades deben confirmarse contra la definición del proceso que produjo el CSV.

In [ ]:
columnas = list(muestra.columns)

def familia_variable(nombre):
    n = nombre.lower()
    if any(t in n for t in ('schema', 'frame', 'duration', 'total_frames', 'time_since')):
        return 'Video y tiempo'
    if any(t in n for t in ('player_id', 'player_name', 'team', 'nombre_norm')):
        return 'Identificación de jugadores/equipo'
    if any(t in n for t in ('ball_position', 'player_position', 'distance_', 'zone')):
        return 'Cancha, posiciones y distancias'
    if any(t in n for t in ('speed', 'velocity', 'acceleration', 'displacement', 'prev_', 'angular')):
        return 'Movimiento y velocidades'
    if any(t in n for t in ('head_', 'shoulder_', 'elbow_', 'wrist_', 'racket_', 'arm_', 'forearm_', 'body_scale', 'pose_')):
        return 'Pose, brazo y raqueta'
    if any(t in n for t in ('implausible', 'available', 'confidence', 'hits_ball')):
        return 'Indicadores y control de calidad'
    if n in ('partido', 'punto'):
        return 'Partido y punto'
    return 'Otras / revisar definición'

def descripcion_inicial(nombre):
    n = nombre.lower()
    if n.endswith('_x') or n.endswith('_y'):
        return 'Componente de coordenada; verificar sistema de referencia y unidad.'
    if 'implausible' in n:
        return 'Indicador asociado a un valor o fila marcado como poco plausible.'
    if 'confidence' in n:
        return 'Medida de confianza del estimador; rango y escala por confirmar.'
    if 'available' in n or 'hits_ball' in n or n == 'wrist_above_head' or n == 'racket_above_shoulder':
        return 'Indicador/condición; codificación por confirmar.'
    if 'speed' in n or 'velocity' in n:
        return 'Medida de velocidad; confirmar unidad y método de cálculo.'
    if 'acceleration' in n:
        return 'Medida de aceleración; confirmar unidad y método de cálculo.'
    if 'angle' in n or 'orientation' in n or 'direction' in n:
        return 'Ángulo u orientación; confirmar referencia y unidad.'
    if 'distance' in n or 'length' in n or 'displacement' in n:
        return 'Distancia, longitud o desplazamiento; confirmar unidad.'
    return 'Revisar valores y documentación para precisar su definición.'

diccionario = pd.DataFrame({
    'variable': columnas,
    'familia inferida': [familia_variable(c) for c in columnas],
    'descripción inicial': [descripcion_inicial(c) for c in columnas],
    'tipo inferido en muestra': [str(muestra[c].dtype) for c in columnas],
})
display(diccionario)

## 3. Resumen descriptivo de la muestra

Se muestran estadísticas para las columnas que pandas reconoce como numéricas en la muestra. Si alguna variable numérica aparece como texto, revisa separadores, valores especiales o mezcla de formatos.

In [ ]:
display(muestra.describe(include='all').T)

categoricas_candidatas = [
    c for c in ['player_name', 'team', 'zone', 'racket_direction', 'partido', 'punto', 'nombre_norm']
    if c in muestra.columns
]
for columna in categoricas_candidatas:
    print(f'\n{columna}: valores más frecuentes en la muestra')
    display(muestra[columna].value_counts(dropna=False).head(15).rename('frecuencia').to_frame())

## 4. Perfil completo por bloques

Esta celda recorre el archivo completo sin conservar los bloques en memoria. Calcula filas, faltantes por variable y frecuencias para las categorías seleccionadas. Puede tardar varios minutos dependiendo del equipo.

In [ ]:
categoricas_perfil = [
    c for c in ['player_name', 'team', 'zone', 'racket_direction', 'partido', 'punto', 'nombre_norm']
    if c in columnas
]
faltantes_totales = pd.Series(0, index=columnas, dtype='int64')
frecuencias = {c: Counter() for c in categoricas_perfil}
filas_totales = 0

for bloque in pd.read_csv(
    RUTA_CSV,
    encoding='utf-8-sig',
    chunksize=CHUNK_SIZE,
    low_memory=False,
):
    filas_totales += len(bloque)
    faltantes_totales = faltantes_totales.add(bloque.isna().sum(), fill_value=0).astype('int64')
    for c in categoricas_perfil:
        frecuencias[c].update(bloque[c].fillna('<FALTANTE>').astype(str).value_counts().to_dict())

perfil_faltantes = pd.DataFrame({
    'faltantes': faltantes_totales,
    'porcentaje_faltante': (faltantes_totales / filas_totales * 100).round(2),
}).sort_values('porcentaje_faltante', ascending=False)

print(f'Filas totales: {filas_totales:,}')
print(f'Variables totales: {len(columnas)}')
display(perfil_faltantes)

for c, conteos in frecuencias.items():
    print(f'\n{c}: categorías más frecuentes en todo el archivo')
    display(pd.Series(conteos).sort_values(ascending=False).head(20).rename('frecuencia').to_frame())

## 5. Calidad y patrones iniciales

Los duplicados se revisan sólo en la muestra. Los conteos de filas marcadas y disponibilidad de pose se calculan sobre el archivo completo cuando esas columnas existen.

In [ ]:
print(f'Filas duplicadas exactas en la muestra: {muestra.duplicated().sum():,} de {len(muestra):,}')

columnas_control = [c for c in ['row_implausible', 'player_speed_implausible', 'ball_speed_implausible', 'pose_available'] if c in columnas]
conteos_control = Counter()
for bloque in pd.read_csv(RUTA_CSV, encoding='utf-8-sig', chunksize=CHUNK_SIZE, low_memory=False, usecols=columnas_control):
    for c in columnas_control:
        conteos_control[c] += int(bloque[c].fillna(False).astype(str).str.lower().isin(['true', '1', 'yes', 'si']).sum())

if columnas_control:
    display(pd.DataFrame({
        'filas marcadas como verdadero': pd.Series(conteos_control),
        'porcentaje del total': (pd.Series(conteos_control) / filas_totales * 100).round(2),
    }))
else:
    print('No se encontraron columnas de control esperadas.')

columnas_numericas = muestra.select_dtypes(include='number').columns.tolist()
if columnas_numericas:
    resumen = muestra[columnas_numericas].describe(percentiles=[.01, .25, .5, .75, .99]).T
    display(resumen[['min', '1%', '25%', '50%', '75%', '99%', 'max']].sort_index())
else:
    print('No se infirieron variables numéricas en la muestra.')

## 6. Gráficos exploratorios de la muestra

Los histogramas y barras siguientes se calculan sólo con la muestra. No son resultados definitivos de todo el conjunto.

In [ ]:
preferidas_numericas = [c for c in ['player_speed_mps', 'ball_speed_mps', 'pose_confidence', 'distance_player_to_ball_m'] if c in muestra.columns and pd.api.types.is_numeric_dtype(muestra[c])]
if preferidas_numericas:
    muestra[preferidas_numericas].hist(bins=30, figsize=(12, 8), layout=(-1, 2))
    plt.suptitle('Distribuciones de variables seleccionadas (muestra)')
    plt.tight_layout()
    plt.show()
else:
    print('No se encontraron variables numéricas seleccionadas para graficar.')

if 'zone' in muestra.columns:
    muestra['zone'].fillna('<FALTANTE>').value_counts().head(15).plot(kind='bar', figsize=(10, 4), title='Frecuencia de zone (muestra)')
    plt.ylabel('Filas')
    plt.tight_layout()
    plt.show()

## 7. Preguntas para continuar el análisis

- ¿Qué representa cada fila: un frame, un jugador por frame o una observación distinta?
- ¿En qué sistema de coordenadas están expresadas las posiciones y cuáles son sus unidades?
- ¿Cómo se codifican los valores booleanos y qué significa exactamente cada indicador de implausibilidad?
- ¿Hay identificadores que permitan separar partidos, puntos, jugadores y videos sin ambigüedad?
- ¿Cómo tratar faltantes, filas implausibles y observaciones sin pose antes de modelar?

Con las respuestas se puede completar el diccionario de datos y definir reglas de limpieza antes de construir modelos.